In [ ]:
# https://datalake.viettelcyber.com/gateway/ui/zeppelin/#/notebook/2MF4T3TRC

In [ ]:
%livy.pyspark
#spark.sql("DROP TABLE IF EXISTS bitu_silver_data.deals_allocation_test")

In [ ]:
%livy.pyspark

# 1. Cấu hình hệ thống & Sửa lỗi Broadcast/Metadata
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.sql("REFRESH TABLE crm_raw.sales_accounts")
spark.sql("REFRESH TABLE crm_raw.business_types")
spark.sql("REFRESH TABLE crm_raw.industry_types")

# 2. Xóa bảng cũ để đảm bảo Schema sạch
spark.sql("DROP TABLE IF EXISTS bitu_silver_data.sales_accounts")

# 3. Logic SQL: Lấy bản ghi NEWEST theo updated_at cho từng Account
query = """
    WITH ranked_accounts AS (
        SELECT 
            *,
            -- Xếp hạng để lấy bản ghi mới nhất cho mỗi ID
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) as rank_account
        FROM crm_raw.sales_accounts
    ),
    latest_accounts AS (
        SELECT * FROM ranked_accounts WHERE rank_account = 1
    )
    
    SELECT
        CAST(s.id AS STRING) AS id,
        s.name,
        s.address,
        s.city,
        s.state,
        s.zipcode,
        s.country,
        s.number_of_employees,
        s.annual_revenue,
        s.website,
        CAST(s.owner_id AS STRING) AS owner_id,
        s.custom_field.cf__am,
        CAST(s.phone AS STRING) AS phone,
        CAST(s.open_deals_amount AS DECIMAL(18,2)) AS open_deals_amount,
        CAST(s.open_deals_count AS BIGINT) AS open_deals_count,
        CAST(s.won_deals_amount AS DECIMAL(18,2)) AS won_deals_amount,
        CAST(s.won_deals_count AS BIGINT) AS won_deals_count,
        s.last_contacted,
        s.last_contacted_mode,
        s.facebook,
        s.twitter,
        s.linkedin,
        s.custom_field.cf_tax_code,
        s.custom_field.cf_alias,
        s.custom_field.cf_segment,
        s.custom_field.cf_segment2,
        s.custom_field.cf_segment3,
        TO_DATE(s.custom_field.cf_incorporation_date, 'yyyy-MM-dd') AS cf_incorporation_date,
        s.custom_field.cf_initial_source,
        s.custom_field.cf_warm_up_source,
        s.custom_field.cf_using_soc,
        s.custom_field.cf_vcs_socothers,
        s.custom_field.cf_soc_brand,
        s.custom_field.cf_interested_products,
        s.custom_field.cf_service_level,
        s.custom_field.cf_product_history,
        TO_DATE(s.created_at) AS created_at,
        TO_DATE(s.updated_at) AS updated_at,
        
        -- Tính toán thêm: Số ngày từ lần cuối cập nhật đến nay
        DATEDIFF(CURRENT_DATE(), TO_DATE(s.updated_at)) AS days_since_update,
        s.parent_sales_account_id,
        s.recent_note,
        to_date(s.last_contacted_via_sales_activity) AS last_contacted_via_sales_activity,
        s.last_contacted_sales_activity_mode,
        s.completed_sales_sequences,
        s.active_sales_sequences,
        TO_DATE(s.last_assigned_at) AS last_assigned_at,
        s.is_deleted,
        
        -- Ép kiểu chuỗi các cột phức tạp để tránh lỗi Job Aborted
        CAST(s.team_user_ids AS STRING) AS team_user_ids,
        CAST(s.domains AS STRING) AS domains,
        CAST(s.tags AS STRING) AS tags,
        
        s.record_type_id,
        s.description,
        s.note,
        s.health_score,
        s.account_tier,
        TO_DATE(s.renewal_date) AS renewal_date,
        
        -- Dữ liệu Join
        b.name AS business_type,
        j.name AS industry_type
        
    FROM latest_accounts s
    LEFT JOIN crm_raw.business_types b ON s.business_type_id = b.id
    LEFT JOIN crm_raw.industry_types j ON s.industry_type_id = j.id
"""

# 4. Thực thi và ghi bảng
try:
    df_sales_accounts = spark.sql(query)
    
    # Ghi vào Silver (Sử dụng format Parquet cho ổn định)
    df_sales_accounts.write \
        .mode("overwrite") \
        .format("parquet") \
        .saveAsTable("bitu_silver_data.sales_accounts")
    
    # Refresh Catalog để các câu lệnh SQL sau đó nhận diện được bảng
    spark.catalog.refreshTable("bitu_silver_data.sales_accounts")
    
    print("--------------------------------------------------")
    print("THANH CONG: Bang sales_accounts da duoc update du lieu moi nhat.")
    print("Total rows (unique): " + str(df_sales_accounts.count()))
    print("--------------------------------------------------")

except Exception as e:
    print("LOI PHAT SINH: " + str(e))
    raise e

# 5. Kiểm tra kết quả
spark.sql("""
    SELECT id, name, updated_at, days_since_update, business_type 
    FROM bitu_silver_data.sales_accounts
    ORDER BY updated_at DESC 
    LIMIT 10
""").show(truncate=False)

In [ ]:
%livy.pyspark

# 1. Cấu hình & Refresh Metadata
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.sql("REFRESH TABLE crm_raw.deals")
spark.sql("REFRESH TABLE crm_raw.deal_stages")
spark.sql("REFRESH TABLE crm_raw.cm_contracts")
spark.sql("REFRESH TABLE crm_raw.deal_payment_statuses")

# 2. Xử lý Logic: Lấy bản ghi mới nhất + Tính ngày + Full tất cả các cột
query = """
    WITH ranked_deals AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY updated_at DESC) as rank_deal
        FROM crm_raw.deals
    ),
    latest_deals AS (
        SELECT * FROM ranked_deals WHERE rank_deal = 1
    ),
    latest_contracts AS (
        SELECT 
            custom_field.cf__opp_id,
            custom_field.cf_sign_date,
            ROW_NUMBER() OVER (PARTITION BY custom_field.cf__opp_id ORDER BY updated_at DESC) as rank_contract
        FROM crm_raw.cm_contracts
    )
    
    SELECT 
        CAST(s.id AS STRING) AS id,
        s.name,
        CAST(s.amount AS DECIMAL(18,2)) AS amount,
        CAST(s.base_currency_amount AS DECIMAL(18,2)) AS base_currency_amount,
        -- CỘT CHANGE_RATE MỚI: base_currency_amount / amount
        ROUND(CAST(s.base_currency_amount AS DOUBLE) / NULLIF(CAST(s.amount AS DOUBLE), 0), 2) AS change_rate,
        TO_DATE(s.expected_close) AS expected_close_date,
        TO_DATE(s.closed_date) AS closed_date,
        TO_TIMESTAMP(s.stage_updated_time) AS stage_updated_time,
        
        -- TÍNH TOÁN LEAD TIME THEO NGÀY
        CASE 
            WHEN s.closed_date IS NOT NULL THEN DATEDIFF(TO_DATE(s.closed_date), TO_DATE(s.created_at))
            ELSE NULL 
        END AS days_to_close,
        DATEDIFF(CURRENT_DATE(), TO_DATE(s.updated_at)) AS days_since_last_update,

        -- FULL TRƯỜNG TỪ CUSTOM_FIELD CỦA BẠN
        s.custom_field.cf_lock_revenue,
        s.custom_field.cf_usd_to_vnd,
        s.custom_field.cf_interested_products,
        TO_DATE(s.custom_field.cf_get_live_date) AS cf_get_live_date,
        s.custom_field.cf_periodicity,
        s.custom_field.cf_select_viettel,
        s.custom_field.cf_has_budget,
        s.custom_field.cf_alias,
        s.custom_field.cf_budget,
        s.custom_field.cf_group,
        s.custom_field.cf_segment,
        s.custom_field.cf_segment2,
        s.custom_field.cf_segment3,
        s.custom_field.cf_channel,
        s.custom_field.cf_weight,
        s.custom_field.cf_bidding_required,
        s.custom_field.cf_presales,
        s.custom_field.cf_project_manager,
        s.custom_field.cf_sale_admin,
        s.custom_field.cf_partner,
        s.custom_field.cf_contract,
        s.custom_field.cf_initial_source,
        s.custom_field.cf_warm_up_source,
        s.custom_field.cf__territory AS cf_territory,
        s.custom_field.cf__duration AS cf_duration,
        s.custom_field.cf__create_contract AS cf_create_contract,
        TO_DATE(s.custom_field.cf__expire_date) AS cf_expire_date,
        s.custom_field.cf__company AS cf_company,
        s.custom_field.cf__address AS cf_address,
        s.custom_field.cf__province AS cf_province,
        s.custom_field.cf__country AS cf_country,
        s.custom_field.cf__email AS cf_email,
        s.custom_field.cf__phone AS cf_phone,
        s.custom_field.cf__contact AS cf_contact,
        s.custom_field.cf__currency AS cf_currency,
        s.custom_field.cf__quotations AS cf_quotations,
        s.custom_field.cf__quotation_status AS cf_quotation_status,
        TO_DATE(s.custom_field.cf__fac_date) AS cf_fac_date,
        s.custom_field.cf__am AS cf_am,
        s.custom_field.cf__check_change AS cf_check_change,
        
        -- CÁC TRƯỜNG THÔNG TIN HỆ THỐNG
        s.probability,
        TO_TIMESTAMP(s.updated_at) AS updated_at,
        TO_TIMESTAMP(s.created_at) AS created_at,
        CAST(s.deal_stage_id AS STRING) AS deal_stage_id,
        CAST(s.deal_payment_status_id AS STRING) AS deal_payment_status_id,
        s.age,
        s.recent_note,
        s.completed_sales_sequences,
        s.active_sales_sequences,
        s.upcoming_activities_time,
        to_timestamp(s.last_assigned_at) AS last_assigned_at,
        s.last_contacted_sales_activity_mode,
        s.last_contacted_via_sales_activity,
        s.expected_deal_value,
        s.is_deleted,
        CAST(s.team_user_ids AS STRING) AS team_user_ids,
        s.forecast_category,
        s.deal_prediction,
        s.deal_prediction_last_updated_at,
        s.last_deal_prediction,
        s.has_products,
        CAST(s.products AS STRING) AS products,
        CAST(s.deal_price_adjustments AS STRING) AS deal_price_adjustments,
        s.rotten_days,
        CAST(s.tags AS STRING) AS tags,
        
        s.owner_id,
        s.sales_account_id,
        s.deal_type_id,
        s.deal_reason_id,
        s.currency_id,

        -- DỮ LIỆU TỪ CÁC BẢNG JOIN
        j.name AS deal_payment_status,
        b.name AS deal_stage_name,
        to_date(i.cf_sign_date) AS cf_sign_date

    FROM latest_deals s
    LEFT JOIN crm_raw.deal_stages b ON s.deal_stage_id = b.id
    LEFT JOIN (SELECT * FROM latest_contracts WHERE rank_contract = 1) i 
        ON CAST(s.id AS STRING) = CAST(i.cf__opp_id AS STRING)
    LEFT JOIN crm_raw.deal_payment_statuses j ON s.deal_payment_status_id = j.id
"""

# 3. Ghi dữ liệu
try:
    df_deals = spark.sql(query)
    
    # Drop bảng cũ để đảm bảo schema mới được áp dụng hoàn toàn
    spark.sql("DROP TABLE IF EXISTS bitu_silver_data.deals")
    
    df_deals.write \
        .mode("overwrite") \
        .format("parquet") \
        .saveAsTable("bitu_silver_data.deals")
    
    print("--------------------------------------------------")
    print("THANH CONG: Bang deals da duoc cap nhat day du tat ca cac cot.")
    print("Tong so deals (unique id): " + str(df_deals.count()))
    print("--------------------------------------------------")

except Exception as e:
    print("LOI: " + str(e))
    raise e

# 4. Xem kết quả
spark.sql("SELECT id, name, days_to_close,change_rate, deal_stage_name, cf_sign_date, updated_at FROM bitu_silver_data.deals LIMIT 10").show()

In [ ]:
%livy.pyspark
from pyspark.sql.functions import from_json, col, posexplode_outer, coalesce, when, trim, lit, current_date, datediff, to_date, row_number
from pyspark.sql.types import ArrayType, StringType, StructType, StructField
from pyspark.sql.window import Window

# 1) Cấu hình
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.sql("REFRESH TABLE crm_raw.deals")
spark.sql("REFRESH TABLE crm_raw.deal_stages")

# 2) Query dữ liệu thô & Lấy bản ghi mới nhất theo deal_id
df_raw_base = spark.sql("""
    SELECT
        CAST(s.id AS STRING) AS deal_id,
        s.name AS deal_name,
        b.name AS deal_stage_name,
        s.custom_field.cf__company AS cf_company,
        -- CỘT CHANGE_RATE MỚI: base_currency_amount / amount
        ROUND(CAST(s.base_currency_amount AS DOUBLE) / NULLIF(CAST(s.amount AS DOUBLE), 0), 2) AS change_rate,
        s.custom_field.cf__am AS cf_am,
        s.custom_field.cf__fac_date AS cf_fac_date,
        s.custom_field.cf__products AS raw_cf_products,
        s.custom_field.cf__allocated_products AS raw_cf_allocated_products,
        s.custom_field.cf__allocated_records AS raw_cf_allocated_records,
        s.created_at,
        s.closed_date,
        s.updated_at
    FROM crm_raw.deals s
    INNER JOIN crm_raw.deal_stages b ON s.deal_stage_id = b.id
""")

window_spec = Window.partitionBy("deal_id").orderBy(col("updated_at").desc())
df_raw = df_raw_base.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1) \
                    .drop("rn")

# 3) Định nghĩa Schemas Đầy Đủ (Để không bị lỗi null khi truy vấn các trường sâu)
record_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("territory", StringType(), True),
    StructField("totalVcsValue", StringType(), True),
    StructField("totalValue", StringType(), True),
    StructField("vcsValue", StringType(), True),
    StructField("forecastValue", StringType(), True),
    StructField("actualValue", StringType(), True),
    StructField("allocationOverrides", StringType(), True)
]))

allo_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("allocationValue", StringType(), True),
    StructField("type", StringType(), True),
    StructField("allocationDuration", StringType(), True),
    StructField("coefficient", StringType(), True),
    StructField("forecastDate", StringType(), True),
    StructField("actualDate", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("productType", StringType(), True),
    StructField("spdvType", StringType(), True),
    StructField("region", StringType(), True)
]))

prod_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("license", StringType(), True),
    StructField("quantitative", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("package", StringType(), True),
    StructField("priceType", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("isQuantityBased", StringType(), True),
    StructField("vat", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("discountType", StringType(), True),
    StructField("basePrice", StringType(), True),
    StructField("baseTotal", StringType(), True),
    StructField("finalTotal", StringType(), True)
]))

# 4) Parse JSON
df_parsed = df_raw.withColumn("rec_arr", from_json(col("raw_cf_allocated_records"), record_schema)) \
                  .withColumn("allo_arr", from_json(col("raw_cf_allocated_products"), allo_schema)) \
                  .withColumn("prod_arr", from_json(col("raw_cf_products"), prod_schema))

# 5) Bung dòng
df_exploded = df_parsed.select("*", posexplode_outer(col("allo_arr")).alias("pos", "dummy_allo"))

# 6) Trích xuất và ép kiểu dưz l
df_final = df_exploded.select(
    col("deal_id"),
    col("deal_name"),
    col("deal_stage_name"),
    col("cf_company"),
    col("cf_am"),
    to_date(col("cf_fac_date"), "yyyy-MM-dd").alias("cf_fac_date"),
    col("pos"),
    
    # --- CỘT CHUNG ---
    coalesce(col("rec_arr")[col("pos")]["id"], col("allo_arr")[col("pos")]["id"], col("prod_arr")[col("pos")]["id"]).alias("cf_id"),
    coalesce(col("rec_arr")[col("pos")]["name"], col("allo_arr")[col("pos")]["name"], col("prod_arr")[col("pos")]["name"]).alias("cf_name"),
    coalesce(col("rec_arr")[col("pos")]["category"], col("allo_arr")[col("pos")]["category"], col("prod_arr")[col("pos")]["category"]).alias("cf_category"),

    # --- ALLOCATED RECORDS ---
    col("rec_arr")[col("pos")]["territory"].alias("cf_territory"),
    col("rec_arr")[col("pos")]["totalVcsValue"].cast("decimal(18,2)").alias("cf_totalvcsvalue"),
    col("rec_arr")[col("pos")]["totalValue"].cast("decimal(18,2)").alias("cf_totalvalue"),
    col("rec_arr")[col("pos")]["vcsValue"].cast("decimal(18,2)").alias("cf_vcsvalue"),
    col("rec_arr")[col("pos")]["forecastValue"].cast("decimal(18,2)").alias("cf_forecastvalue"),
    col("rec_arr")[col("pos")]["actualValue"].cast("decimal(18,2)").alias("cf_actualvalue"),
    col("rec_arr")[col("pos")]["allocationOverrides"].alias("cf_allocationoverrides"),

    # --- ALLOCATED PRODUCTS ---
    col("allo_arr")[col("pos")]["allocationValue"].cast("decimal(18,2)").alias("cf_allocationvalue"),
    col("allo_arr")[col("pos")]["type"].alias("cf_allocationtype"),
    col("allo_arr")[col("pos")]["allocationDuration"].alias("cf_allocationduration"),
    col("allo_arr")[col("pos")]["coefficient"].alias("cf_coefficient"),
    to_date(col("allo_arr")[col("pos")]["forecastDate"], "dd/MM/yyyy").alias("cf_forecastdate"),
    to_date(col("allo_arr")[col("pos")]["actualDate"], "dd/MM/yyyy").alias("cf_actualdate"),
    col("allo_arr")[col("pos")]["productType"].alias("cf_producttype"),
    col("allo_arr")[col("pos")]["spdvType"].alias("cf_spdvtype"),
    col("allo_arr")[col("pos")]["region"].alias("cf_region"),
    when(trim(col("allo_arr")[col("pos")]["currency"]) == "đ", "VND")
        .when(trim(col("allo_arr")[col("pos")]["currency"]) == "$", "USD")
        .otherwise(col("allo_arr")[col("pos")]["currency"]).alias("cf_currency"),

    # --- PRODUCTS ---
    col("prod_arr")[col("pos")]["license"].alias("cf_license"),
    col("prod_arr")[col("pos")]["quantitative"].alias("cf_quantitative"),
    col("prod_arr")[col("pos")]["unit"].alias("cf_unit"),
    col("prod_arr")[col("pos")]["package"].alias("cf_package"),
    col("prod_arr")[col("pos")]["priceType"].alias("cf_pricetype"),
    col("prod_arr")[col("pos")]["duration"].alias("cf_duration"),
    col("prod_arr")[col("pos")]["isQuantityBased"].alias("cf_isquantitybased"),
    col("prod_arr")[col("pos")]["vat"].cast("decimal(18,2)").alias("cf_vat"),
    col("prod_arr")[col("pos")]["discount"].cast("decimal(18,2)").alias("cf_discount"),
    col("prod_arr")[col("pos")]["discountType"].alias("cf_discounttype"),
    col("prod_arr")[col("pos")]["basePrice"].cast("decimal(18,2)").alias("cf_baseprice"),
    col("prod_arr")[col("pos")]["baseTotal"].cast("decimal(18,2)").alias("cf_basetotal"),
    col("prod_arr")[col("pos")]["finalTotal"].cast("decimal(18,2)").alias("cf_finaltotal"),

    # --- THỜI GIAN ---
    to_date(col("created_at")).alias("created_at"),
    to_date(col("updated_at")).alias("updated_at"),
    datediff(current_date(), col("updated_at")).alias("days_since_last_update")
).filter(col("cf_id").isNotNull())

# 7) Loại bỏ trùng lặp & Tính toán cột bổ sung
df_final = df_final.dropDuplicates(["deal_id", "pos"]) \
                   .withColumn("first_payment_date", coalesce(col("cf_fac_date"), col("cf_forecastdate")))

# 8) Lowercase headers
for col_name in df_final.columns:
    df_final = df_final.withColumnRenamed(col_name, col_name.lower())

# 9) Lưu Silver Table
df_final.write.mode("overwrite").saveAsTable("bitu_silver_data.deals_allocation")

# Hiển thị kiểm tra
spark.sql("SELECT * FROM bitu_silver_data.deals_allocation LIMIT 20").show()

In [ ]:
%livy.pyspark
from pyspark.sql.functions import from_json, col, posexplode_outer, coalesce, when, trim, lit, current_date, datediff, to_date, row_number, round
from pyspark.sql.types import ArrayType, StringType, StructType, StructField
from pyspark.sql.window import Window

# 1) Cấu hình
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.sql("REFRESH TABLE crm_raw.deals")
spark.sql("REFRESH TABLE crm_raw.deal_stages")

# 2) Query dữ liệu thô & Lấy bản ghi mới nhất theo deal_id
df_raw_base = spark.sql("""
    SELECT
        CAST(s.id AS STRING) AS deal_id,
        s.name AS deal_name,
        b.name AS deal_stage_name,
        s.custom_field.cf__company AS cf_company,
        -- CỘT CHANGE_RATE: base_currency_amount / amount
        ROUND(CAST(s.base_currency_amount AS DOUBLE) / NULLIF(CAST(s.amount AS DOUBLE), 0), 2) AS change_rate,
        s.custom_field.cf__am AS cf_am,
        s.custom_field.cf__fac_date AS cf_fac_date,
        s.custom_field.cf__products AS raw_cf_products,
        s.custom_field.cf__allocated_products AS raw_cf_allocated_products,
        s.custom_field.cf__allocated_records AS raw_cf_allocated_records,
        s.created_at,
        s.closed_date,
        s.updated_at
    FROM crm_raw.deals s
    INNER JOIN crm_raw.deal_stages b ON s.deal_stage_id = b.id
""")

window_spec = Window.partitionBy("deal_id").orderBy(col("updated_at").desc())
df_raw = df_raw_base.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1) \
                    .drop("rn")

# 3) Định nghĩa Schemas
record_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("territory", StringType(), True),
    StructField("totalVcsValue", StringType(), True),
    StructField("totalValue", StringType(), True),
    StructField("vcsValue", StringType(), True),
    StructField("forecastValue", StringType(), True),
    StructField("actualValue", StringType(), True),
    StructField("allocationOverrides", StringType(), True)
]))

allo_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("allocationValue", StringType(), True),
    StructField("type", StringType(), True),
    StructField("allocationDuration", StringType(), True),
    StructField("coefficient", StringType(), True),
    StructField("forecastDate", StringType(), True),
    StructField("actualDate", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("productType", StringType(), True),
    StructField("spdvType", StringType(), True),
    StructField("region", StringType(), True)
]))

prod_schema = ArrayType(StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("license", StringType(), True),
    StructField("quantitative", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("package", StringType(), True),
    StructField("priceType", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("isQuantityBased", StringType(), True),
    StructField("vat", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("discountType", StringType(), True),
    StructField("basePrice", StringType(), True),
    StructField("baseTotal", StringType(), True),
    StructField("finalTotal", StringType(), True)
]))

# 4) Parse JSON
df_parsed = df_raw.withColumn("rec_arr", from_json(col("raw_cf_allocated_records"), record_schema)) \
                  .withColumn("allo_arr", from_json(col("raw_cf_allocated_products"), allo_schema)) \
                  .withColumn("prod_arr", from_json(col("raw_cf_products"), prod_schema))

# 5) Bung dòng
df_exploded = df_parsed.select("*", posexplode_outer(col("allo_arr")).alias("pos", "dummy_allo"))

# 6) Trích xuất và ép kiểu dữ liệu
df_final = df_exploded.select(
    col("deal_id"),
    col("deal_name"),
    col("deal_stage_name"),
    col("cf_company"),
    col("cf_am"),
    col("change_rate"), # Giữ lại để tính toán ở bước sau
    to_date(col("cf_fac_date"), "yyyy-MM-dd").alias("cf_fac_date"),
    col("pos"),
    
    # --- CỘT CHUNG ---
    coalesce(col("rec_arr")[col("pos")]["id"], col("allo_arr")[col("pos")]["id"], col("prod_arr")[col("pos")]["id"]).alias("cf_id"),
    coalesce(col("rec_arr")[col("pos")]["name"], col("allo_arr")[col("pos")]["name"], col("prod_arr")[col("pos")]["name"]).alias("cf_name"),
    coalesce(col("rec_arr")[col("pos")]["category"], col("allo_arr")[col("pos")]["category"], col("prod_arr")[col("pos")]["category"]).alias("cf_category"),

    # --- ALLOCATED RECORDS ---
    col("rec_arr")[col("pos")]["territory"].alias("cf_territory"),
    col("rec_arr")[col("pos")]["totalVcsValue"].cast("decimal(18,2)").alias("cf_totalvcsvalue"),
    col("rec_arr")[col("pos")]["totalValue"].cast("decimal(18,2)").alias("cf_totalvalue"),
    col("rec_arr")[col("pos")]["vcsValue"].cast("decimal(18,2)").alias("cf_vcsvalue"),
    col("rec_arr")[col("pos")]["forecastValue"].cast("decimal(18,2)").alias("cf_forecastvalue"),
    col("rec_arr")[col("pos")]["actualValue"].cast("decimal(18,2)").alias("cf_actualvalue"),
    col("rec_arr")[col("pos")]["allocationOverrides"].alias("cf_allocationoverrides"),

    # --- ALLOCATED PRODUCTS ---
    col("allo_arr")[col("pos")]["allocationValue"].cast("decimal(18,2)").alias("cf_allocationvalue"),
    col("allo_arr")[col("pos")]["type"].alias("cf_allocationtype"),
    col("allo_arr")[col("pos")]["allocationDuration"].alias("cf_allocationduration"),
    col("allo_arr")[col("pos")]["coefficient"].alias("cf_coefficient"),
    to_date(col("allo_arr")[col("pos")]["forecastDate"], "dd/MM/yyyy").alias("cf_forecastdate"),
    to_date(col("allo_arr")[col("pos")]["actualDate"], "dd/MM/yyyy").alias("cf_actualdate"),
    col("allo_arr")[col("pos")]["productType"].alias("cf_producttype"),
    col("allo_arr")[col("pos")]["spdvType"].alias("cf_spdvtype"),
    col("allo_arr")[col("pos")]["region"].alias("cf_region"),
    when(trim(col("allo_arr")[col("pos")]["currency"]) == "đ", "VND")
        .when(trim(col("allo_arr")[col("pos")]["currency"]) == "$", "USD")
        .otherwise(col("allo_arr")[col("pos")]["currency"]).alias("cf_currency"),

    # --- PRODUCTS ---
    col("prod_arr")[col("pos")]["license"].alias("cf_license"),
    col("prod_arr")[col("pos")]["quantitative"].alias("cf_quantitative"),
    col("prod_arr")[col("pos")]["unit"].alias("cf_unit"),
    col("prod_arr")[col("pos")]["package"].alias("cf_package"),
    col("prod_arr")[col("pos")]["priceType"].alias("cf_pricetype"),
    col("prod_arr")[col("pos")]["duration"].alias("cf_duration"),
    col("prod_arr")[col("pos")]["isQuantityBased"].alias("cf_isquantitybased"),
    col("prod_arr")[col("pos")]["vat"].cast("decimal(18,2)").alias("cf_vat"),
    col("prod_arr")[col("pos")]["discount"].cast("decimal(18,2)").alias("cf_discount"),
    col("prod_arr")[col("pos")]["discountType"].alias("cf_discounttype"),
    col("prod_arr")[col("pos")]["basePrice"].cast("decimal(18,2)").alias("cf_baseprice"),
    col("prod_arr")[col("pos")]["baseTotal"].cast("decimal(18,2)").alias("cf_basetotal"),
    col("prod_arr")[col("pos")]["finalTotal"].cast("decimal(18,2)").alias("cf_finaltotal"),

    # --- THỜI GIAN ---
    to_date(col("created_at")).alias("created_at"),
    to_date(col("updated_at")).alias("updated_at"),
    datediff(current_date(), col("updated_at")).alias("days_since_last_update")
).filter(col("cf_id").isNotNull())

# 7) Loại bỏ trùng lặp & Tính toán cột bổ sung (Bao gồm Base Revenue Amount)
df_final = df_final.dropDuplicates(["deal_id", "pos"]) \
    .withColumn("first_payment_date", coalesce(col("cf_fac_date"), col("cf_forecastdate"))) \
    .withColumn(
        "base_revenue_amount",
        when(col("cf_currency") == 'USD', col("cf_vcsvalue").cast("double") * col("change_rate"))
        .when(col("cf_currency") == 'VND', col("cf_vcsvalue").cast("double"))
        .otherwise(col("cf_vcsvalue").cast("double"))
        .cast("decimal(18,2)")
    )

# 8) Lowercase headers
for col_name in df_final.columns:
    df_final = df_final.withColumnRenamed(col_name, col_name.lower())

# 9) Lưu Silver Table
df_final.write.mode("overwrite").saveAsTable("bitu_silver_data.deals_allocation_test")

# Hiển thị kiểm tra
spark.sql("SELECT deal_id, cf_currency, change_rate, cf_vcsvalue, base_revenue_amount FROM bitu_silver_data.deals_allocation_test LIMIT 20").show()

In [ ]:
%livy.pyspark
from pyspark.sql.functions import sum as _sum, round as _round

# 1. Tính tổng các phần đã phân bổ theo từng deal_id
df_check = spark.sql("""
    SELECT 
        deal_id, 
        MAX(base_currency_amount) as original_amount,
        SUM(cf_allocated_base_amount) as total_allocated_amount
    FROM bitu_silver_data.deals_allocation_test
    GROUP BY deal_id
""")

# 2. Tính toán độ lệch (Difference)
# Làm tròn đến 2 chữ số thập phân để tránh sai số dấu phẩy động (floating point)
df_diff = df_check.withColumn("diff", 
    _round(col("original_amount") - col("total_allocated_amount"), 2)
)

# 3. Hiển thị các trường hợp bị lệch (nếu có)
error_count = df_diff.filter(col("diff") != 0).count()

if error_count == 0:
    print("✅ Tuyệt vời! Toàn bộ {0} deals đã được phân bổ chính xác 100%.".format(df_diff.count()))
else:
    print("⚠️ Cảnh báo: Có {0} deals bị lệch số liệu sau khi phân bổ:".format(error_count))
    df_diff.filter(col("diff") != 0).show()

# 4. Thống kê tổng quan
spark.sql("""
    SELECT 
        'Kiểm tra tổng' as status,
        SUM(base_currency_amount) as sum_original,
        SUM(cf_allocated_base_amount) as sum_allocated
    FROM bitu_silver_data.deals_allocation_test
""").show()

In [ ]:
%livy.pyspark

# -------------------------------------------------------------------------
# 1) DỌN DẸP BẢNG CŨ
# -------------------------------------------------------------------------
spark.sql("DROP TABLE IF EXISTS bitu_silver_data.pricebook")

# -------------------------------------------------------------------------
# 2) TRUY VẤN VÀ BIẾN ĐỔI DỮ LIỆU (TRANSFORMATION)
# -------------------------------------------------------------------------
# Sử dụng DISTINCT để loại bỏ dữ liệu trùng lặp
# Ép kiểu (CAST) một số trường quan trọng để chuẩn hóa dữ liệu tầng Silver
df_pricebooks = spark.sql("""
    SELECT DISTINCT
        CAST(s.id AS STRING)            AS product_id,
        b.name                          AS category_name,
        b.custom_field.cf_category      AS category,
        b.custom_field.cf_version       AS version,
        b.custom_field.cf_item_type     AS item_type,
        b.custom_field.cf_active        AS cf_active,
        s.name                          AS product_name,
        s.owner_id,
        s.custom_field.cf_sku           AS sku,
        s.custom_field.cf_type          AS type,
        s.custom_field.cf_sub_type      AS sub_type,
        s.custom_field.cf_license       AS license,
        s.custom_field.cf_price_type    AS price_type,
        s.custom_field.cf_package       AS package,
        CAST(s.custom_field.cf_price AS DECIMAL(18,2)) AS price,
        s.custom_field.cf_currency      AS currency,
        s.custom_field.cf_min           AS min_val,
        s.custom_field.cf_max           AS max_val,
        s.custom_field.cf_unit          AS unit,
        s.custom_field.cf_is_quantity_based,
        s.custom_field.cf_is_related_csmp,
        s.custom_field.cf_csmp_discount,
        s.custom_field.cf_csmp_discount_silver,
        s.custom_field.cf_csmp_discount_gold,
        s.custom_field.cf_csmp_discount_diamond,
        s.custom_field.cf_catalog,
        s.custom_field.cf_related_pricebook,
        TO_DATE(s.created_at)           AS created_at,
        s.creator_id,
        TO_DATE(s.updated_at)           AS updated_at,
        s.updater_id
    FROM crm_raw.cm_pricebook s
    LEFT JOIN crm_raw.cm_catalog b ON s.custom_field.cf_catalog = b.id
""")

# -------------------------------------------------------------------------
# 3) KIỂM TRA DỮ LIỆU TRONG DATAFRAME TRƯỚC KHI GHI
# -------------------------------------------------------------------------
print("Preview dữ liệu trước khi lưu:")
df_pricebooks.show(10, truncate=False)

# -------------------------------------------------------------------------
# 4) LƯU DỮ LIỆU VÀO SILVER LAYER
# -------------------------------------------------------------------------
# Sử dụng format("parquet") vì môi trường của bạn chưa cài đặt Delta Lake.
# Parquet là định dạng lưu trữ cột (columnar) tối ưu cho truy vấn phân tích.
df_pricebooks.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("bitu_silver_data.pricebook")

print("Đã lưu bảng bitu_silver_data.pricebook thành công.")

# -------------------------------------------------------------------------
# 5) KIỂM TRA BẢNG SAU KHI GHI
# -------------------------------------------------------------------------
print("Truy vấn kiểm tra từ bảng đã lưu:")
spark.sql("SELECT * FROM bitu_silver_data.pricebook LIMIT 10").show(truncate=False)

In [ ]:
%livy.pyspark
from pyspark.sql.functions import col, lit, to_date, when, expr, current_timestamp, explode, udf
from pyspark.sql.types import ArrayType, IntegerType
import pyspark.sql.functions as F

# 1) Đọc dữ liệu
df = spark.sql("SELECT * FROM bitu_silver_data.deals_allocation WHERE cf_vcsvalue IS NOT NULL")

# 2) UDF sinh mảng index
def generate_payment_index(n):
    if n is None or n < 1:
        return [1]
    return list(range(1, int(n) + 1))

generate_payment_index_udf = udf(generate_payment_index, ArrayType(IntegerType()))

# 3) Xử lý Explode và tính Date
# Thêm current_timestamp ở bước này để dùng so sánh ngay sau đó
df_processed = df.withColumn("payment_array", generate_payment_index_udf(col("cf_allocationduration"))) \
                 .withColumn("p_index", explode(col("payment_array"))) \
                 .withColumn("update_at", current_timestamp())

df_with_dates = df_processed.withColumn(
    "payment_date", 
    expr("add_months(to_date(first_payment_date), p_index - 1)")
)

# 4) Phân loại Actual và Forecast Revenue
# Logic: Nếu payment_date <= update_at -> Actual, ngược lại -> Forecast
df_final = df_with_dates.withColumn(
    "actual_revenue",
    when(col("payment_date") <= col("update_at"), col("cf_vcsvalue").cast("double")).otherwise(lit(0))
).withColumn(
    "forecast_revenue",
    when(col("payment_date") > col("update_at"), col("cf_vcsvalue").cast("double")).otherwise(lit(0))
)

# 5) Select các cột cần thiết
df_result = df_final.select(
    col("payment_date"),
    col("update_at"),
    col("actual_revenue"),
    col("forecast_revenue"), # Cột mới thêm vào
    col("cf_producttype"),
    col("cf_name"),
    col("cf_category"),
    col("cf_territory"),
    col("cf_license"),
    col("deal_id").alias("id"),
    col("cf_id"),
    col("cf_allocationduration").cast("bigint"),
    col("cf_currency"),
    col("deal_stage_name"),
    col("p_index").cast("bigint").alias("payment_index")
)

# 6) Lưu và hiển thị
df_result.write.mode("overwrite").saveAsTable("bitu_silver_data.revenue")
df_result.show(5)

In [ ]:
%livy.pyspark
from pyspark.sql.functions import col, lit, to_date, when, expr, current_timestamp, explode, udf
from pyspark.sql.types import ArrayType, IntegerType
import pyspark.sql.functions as F

# 1) Đọc dữ liệu
# Lưu ý: Bảng deals cần có cột change_rate bạn vừa tạo ở bước trước
df_deals = spark.sql("SELECT id, change_rate FROM bitu_silver_data.deals")
df_allocation = spark.sql("SELECT * FROM bitu_silver_data.deals_allocation WHERE cf_vcsvalue is NOT NULL")

# Join để lấy change_rate sang bảng allocation
df = df_allocation.join(df_deals, df_allocation.deal_id == df_deals.id, "left")

# 2) UDF sinh mảng index
def generate_payment_index(n):
    if n is None or n < 1:
        return [1]
    return list(range(1, int(n) + 1))

generate_payment_index_udf = udf(generate_payment_index, ArrayType(IntegerType()))

# 3) Xử lý Explode và tính toán thời gian
df_processed = df.withColumn("payment_array", generate_payment_index_udf(col("cf_allocationduration"))) \
                 .withColumn("p_index", explode(col("payment_array"))) \
                 .withColumn("update_at", current_timestamp())

df_with_payment_date = df_processed.withColumn(
    "calculated_payment_date", 
    expr("add_months(to_date(first_payment_date), p_index - 1)")
)

# 4) Xử lý logic Revenue và Flag is_actual
df_final = df_with_payment_date.withColumn(
    # Tính base_revenue_amount dựa trên loại tiền tệ
    "base_revenue_amount",
    when(col("cf_currency") == 'USD', col("cf_vcsvalue").cast("double") * col("change_rate"))
    .when(col("cf_currency") == 'VND', col("cf_vcsvalue").cast("double"))
    .otherwise(col("cf_vcsvalue").cast("double")) # Default nếu là loại tiền khác
).withColumn(
    # Check thực tế hay dự báo
    "is_actual",
    when(col("calculated_payment_date") <= col("update_at"), True).otherwise(False)
)

# 5) Select và định dạng kết quả cuối cùng
df_result = df_final.select(
    col("calculated_payment_date").alias("payment_date"),
    col("update_at"),
    col("cf_vcsvalue").cast("double").alias("revenue_amount"),
    col("base_revenue_amount"),
    col("is_actual"),
    col("cf_producttype"),
    col("cf_name"),
    col("cf_category"),
    col("cf_territory"),
    col("cf_license"),
    col("deal_id").alias("id"),
    col("cf_id"),
    col("cf_allocationduration").cast("bigint"),
    col("cf_currency"),
    col("deal_stage_name"),
    col("p_index").cast("bigint").alias("payment_index")
)

# 6) Lưu và hiển thị
df_result.write.mode("overwrite").saveAsTable("bitu_silver_data.payment_records")
df_result.show(15)

In [ ]:
%livy.pyspark

# 1. Cấu hình xử lý múi giờ và định dạng thời gian cũ
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 2. Đọc dữ liệu thô
df_raw = spark.read.parquet('/opt/datasets/DWS/VCS_Sales/trino_kinhdoanh')
df_raw.createOrReplaceTempView("raw_data")

# 3. Xóa bảng cũ
spark.sql("DROP TABLE IF EXISTS bitu_silver_data.kinhdoanh")

# 4. Tạo bảng với logic ép kiểu qua Timestamp trung gian
# Chúng ta dùng CAST(date/1000000 AS TIMESTAMP) nếu là Microseconds 
# Hoặc đơn giản nhất là dùng hàm của Spark để tự nhận diện
spark.sql("""
  CREATE TABLE bitu_silver_data.kinhdoanh
  USING PARQUET
  LOCATION '/opt/datasets/DWS/VCS_Sales/trino_kinhdoanh_final'
  AS
  SELECT 
        license,
        CAST(revenue AS DOUBLE) as revenue,
        company_name,
        category_code,
        -- BƯỚC QUAN TRỌNG: Chuyển BigInt sang Timestamp trước, sau đó mới sang Date
        -- Dùng to_date trên cột kiểu Long (L) thường cần qua bước chuyển đổi
        to_date(CAST(date/1000000000 AS TIMESTAMP)) as report_date, 
        customer,
        segment,
        producttype,
        channel,
        is_SOC,
        group_spdv,
        department,
        old_category,
        category,
        segment3,
        CAST(month AS INT) as month,
        CAST(VAT AS DOUBLE) as vat,
        payment_stage,
        AM,
        Presale
  FROM raw_data
""")

# 5. Kiểm tra kết quả
spark.sql("SELECT * FROM bitu_silver_data.kinhdoanh LIMIT 5").show()

In [ ]:
%livy.pyspark

# 1. Cấu hình
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# --- XỬ LÝ FILE 1: PLAN_FIN ---
df_fin = spark.read.parquet('/opt/datasets/DWS/VCS_Sales/trino_plankinhdoanh/plan_FIN.parquet')
df_fin.createOrReplaceTempView("raw_fin")

spark.sql("DROP TABLE IF EXISTS bitu_silver_data.plan_fin")
spark.sql("""
  CREATE TABLE bitu_silver_data.plan_fin
  USING PARQUET
  AS
  SELECT 
        TO_DATE(CAST(plan_date/1000000000 AS TIMESTAMP)) as plan_date, 
        CAST(company AS STRING) as company,
        CAST(viettel_group AS DECIMAL(18,2)) as plan_viettel_group,
        CAST(plan_must AS DECIMAL(18,2)) as plan_must,
        CAST(plan_nice AS DECIMAL(18,2)) as plan_nice,
        CAST(segment AS STRING) as segment
  FROM raw_fin
""")

# --- XỬ LÝ FILE 2: PLAN_SPDV_KD ---
df_spdv = spark.read.parquet('/opt/datasets/DWS/VCS_Sales/trino_plankinhdoanh/plan_SPDV_KD.parquet')
df_spdv.createOrReplaceTempView("raw_spdv")

spark.sql("DROP TABLE IF EXISTS bitu_silver_data.plan_spdv_kd")
spark.sql("""
  CREATE TABLE bitu_silver_data.plan_spdv_kd
  USING PARQUET
  AS
  SELECT 
        TO_DATE(CAST(plan_date/1000000000 AS TIMESTAMP)) as plan_date, 
        CAST(plan_must AS DECIMAL(18,2)) as plan_must,
        CAST(plan_nice AS DECIMAL(18,2)) as plan_nice,
        CAST(category AS STRING) as category,
        CAST(category_code AS STRING) as category_code
  FROM raw_spdv
""")

# Kiểm tra kết quả
spark.sql("SELECT 'plan_fin' as tbl, count(*) FROM bitu_silver_data.plan_fin").show()
spark.sql("SELECT 'plan_spdv_kd' as tbl, count(*) FROM bitu_silver_data.plan_spdv_kd").show()

In [ ]:
%livy.pyspark

# 1. Cấu hình hệ thống
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# Định nghĩa folder gốc
base_path = "/opt/datasets/DWS/VCS_Sales/trino_convensiondata"

# 2. Đảm bảo Database tồn tại
spark.sql("CREATE DATABASE IF NOT EXISTS bitu_schema")

# --- XỬ LÝ FILE 1: TABLE_BUSINESS_CONVENTION ---
# Dùng cách cộng chuỗi để tránh lỗi Syntax f-string
file_table = base_path + "/table_business_convention.parquet"
df_table = spark.read.parquet(file_table)
df_table.createOrReplaceTempView("raw_table_convention")

# Dùng DROP và CREATE AS để đảm bảo tính tương thích cao
spark.sql("DROP TABLE IF EXISTS bitu_schema.table_business_convention")
spark.sql("""
    CREATE TABLE bitu_schema.table_business_convention
    USING PARQUET
    AS
    SELECT * FROM raw_table_convention
""")

# --- XỬ LÝ FILE 2: COLUMN_BUSINESS_CONVENTION ---
file_column = base_path + "/column_business_convention.parquet"
df_column = spark.read.parquet(file_column)
df_column.createOrReplaceTempView("raw_column_convention")

spark.sql("DROP TABLE IF EXISTS bitu_schema.column_business_convention")
spark.sql("""
    CREATE TABLE bitu_schema.column_business_convention
    USING PARQUET
    AS
    SELECT * FROM raw_column_convention
""")

# --- KIỂM TRA KẾT QUẢ ---
spark.sql("SELECT 'table_convention' as tbl, count(*) as total FROM bitu_schema.table_business_convention").show()
spark.sql("SELECT 'column_convention' as tbl, count(*) as total FROM bitu_schema.column_business_convention").show()

In [ ]:
%livy.pyspark
from pyspark.sql import Row

# 1. Thiết lập cấu hình
db_name = "bitu_silver_data"
output_path = "/opt/datasets/DWS/bitu_silver_schema_csv"

# 2. Lấy danh sách bảng
tables = spark.catalog.listTables(db_name)

# 3. Thu thập thông tin schema vào một danh sách các Row
schema_rows = []

for table in tables:
    table_name = table.name
    full_table_name = "{0}.{1}".format(db_name, table_name)
    
    try:
        df_table = spark.table(full_table_name)
        for field in df_table.schema:
            # Tạo Row để chuyển thành DataFrame sau này, thêm thông tin về nullable
            schema_rows.append(Row(
                db_name=db_name,
                table_name=table_name, 
                field_name=field.name, 
                data_type=field.dataType.simpleString(),
                nullable=field.nullable  # Thêm cột nullable
            ))
    except Exception as e:
        print("Lỗi khi đọc bảng {0}".format(table_name))

# 4. Tạo DataFrame từ danh sách Row
# Sắp xếp theo Ten_Bang để các cột của cùng một bảng nằm cạnh nhau
schema_df = spark.createDataFrame(schema_rows) \
    .select("db_name", "table_name", "field_name", "data_type", "nullable") \
    .orderBy("db_name")

# 5. Lưu kết quả ra CSV (Excel có thể đọc được)
# .coalesce(1) đảm bảo dữ liệu chỉ ghi ra 1 file duy nhất thay vì nhiều mảnh
schema_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv(output_path)

print("Đã xuất schema ra folder: {0}".format(output_path))
